# React — Architecture

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> This topic has no playground experiment. Nothing here is a React behaviour to observe — it is
> a set of decisions you make about code you have already learned to write. The runnable cells
> measure the consequences of those decisions instead.

## LESSON 65 — Organising by feature

You have built three projects. Each one started with five files and ended with twenty, and
somewhere in the middle you had to decide where things go. This lesson is about that decision.

Start with the honest answer, from React's own documentation:

> React doesn't have opinions on how you put files into folders.

That is not evasion — it is a real fact about React. There is no `pages/` convention, no
required `components/` folder, nothing the build tool looks for. Every structure you have seen
online is somebody's preference.

### The two structures everyone actually uses

**By file type.** One folder per kind of thing: `components/`, `hooks/`, `services/`,
`utils/`. It looks tidy, and it is what most small projects drift into — including yours.

**By feature.** One folder per part of the product: `features/expenses/`, `features/reports/`,
each holding its own components, hooks, service and helpers.

The argument for feature folders is not aesthetic, and you do not have to take it on faith —
the example cell below measures it. A change is almost never "all the components" or "all the
hooks". It is "everything to do with expenses". By type, that work is spread across four
folders; by feature, it is one folder, and you can see the whole change at once.

The principle behind it has a name — **colocation**: files that change together should sit
together.

### What about the things that belong to nobody

`Button.jsx`, `formatMoney.js`, the date helper. They do not belong to a feature, so they go in
a shared folder — `src/shared/`, or `src/components/ui/` and `src/lib/`. Naming varies; what
matters is the rule for *moving* something there:

> **Move a file to shared when the second feature imports it.** Not before.

The first feature keeps it. The moment a second one needs it, it is genuinely shared and the
move is obvious — and you have avoided inventing a shared API for a single caller.

### Two warnings from the same source

> …don't nest folders more than three or four levels deep

Deep trees make relative imports painful and moving a file a chore.

And, most usefully:

> …don't spend more than five minutes on choosing a file structure.

Start with whatever is obvious for the size you are at, keep it flat, and restructure when it
starts to hurt. A project of ten files needs no structure at all.

### Key Notes

- React has no opinion about folders. Nothing in the build depends on your structure.
- By type is fine while small; by feature keeps a change inside one folder as things grow.
- Colocation: files that change together live together.
- Move something to shared when the **second** feature imports it — not in anticipation.

### Example

**Runnable — plain JS.** The argument for feature folders, measured rather than asserted: take
one real change and count the folders you have to open under each layout.

In [ ]:
// L65 — the same project, two layouts, one change

const l65Files = [
  "src/components/ExpenseForm.jsx",
  "src/components/ExpenseList.jsx",
  "src/components/ExpenseRow.jsx",
  "src/components/ReportChart.jsx",
  "src/components/ReportFilters.jsx",
  "src/components/Button.jsx",
  "src/hooks/useExpenses.js",
  "src/hooks/useReport.js",
  "src/services/expenses.js",
  "src/services/report.js",
  "src/utils/validateExpense.js",
  "src/utils/formatMoney.js",
];

const l65FolderOf = (path) => path.split("/").slice(0, -1).join("/");
const l65NameOf = (path) => path.split("/").pop();

// the change: "add a `note` field to an expense"
const l65Change = l65Files.filter((p) => /expense/i.test(l65NameOf(p)));

console.log("files this change touches:", l65Change.length);
console.log("by type — folders to open:", [...new Set(l65Change.map(l65FolderOf))]);

// now regroup the SAME files by feature
function l65FeatureOf(path) {
  const name = l65NameOf(path);
  if (/expense/i.test(name)) return "src/features/expenses";
  if (/report/i.test(name)) return "src/features/reports";
  return "src/shared";                       // belongs to nobody in particular
}

const l65ByFeature = l65Files.map((p) => `${l65FeatureOf(p)}/${l65NameOf(p)}`);

console.log("\nby feature:");
for (const folder of new Set(l65ByFeature.map(l65FolderOf))) {
  console.log(" ", folder);
  for (const f of l65ByFeature.filter((p) => l65FolderOf(p) === folder)) {
    console.log("     ", l65NameOf(f));
  }
}

const l65ChangeByFeature = l65ByFeature.filter((p) => /expense/i.test(l65NameOf(p)));
console.log("\nby feature — folders to open:", [
  ...new Set(l65ChangeByFeature.map(l65FolderOf)),
]);

### Exercise

**Runnable — plain JS.**

1. The app grows a third feature: `BudgetCard.jsx`, `useBudget.js`, `budget.js` and
   `validateBudget.js`. Add them to `l65Files` in their by-type folders, then run both groupings
   again. How many folders does a budget change touch under each layout?
2. Write `l65FoldersTouched(files, keyword)` that takes any file list and a keyword and returns
   the number of distinct folders a change to that keyword would open. Call it for `expense`,
   `report` and `budget` against both layouts, and print a small comparison table.
3. Two files refused to be classified by feature. Name them, say which folder they ended up in,
   and write one sentence stating the rule that would have told you to put them there — and the
   rule for when a *feature* file should move to join them.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** The shared folder is where structure goes wrong, because everything
looks shareable if you squint.

You are given, for each module, the list of features that import it:

```js
const l65Imports = {
  "formatMoney.js":     ["expenses", "reports"],
  "validateExpense.js": ["expenses"],
  "Button.jsx":         ["expenses", "reports", "budgets"],
  "expenseColours.js":  ["expenses"],
  "useReport.js":       ["reports"],
  "dateRange.js":       ["reports", "budgets"],
};
```

1. Write `l65ShouldBeShared(imports)` returning the modules that genuinely belong in `shared/`,
   applying the rule from the lesson. Print the two lists.
2. `expenseColours.js` is imported by one feature but *sounds* generic. In a comment, say what
   moving it to `shared/` would cost, and when you would move it.
3. One more question, and it is the one that matters: what does this rule protect you from that
   "put anything reusable in shared" does not?

In [ ]:
// Your code here

## LESSON 66 — Where state should live

There is a pattern you will meet in every older React codebase and half the tutorials online,
and you should know both what it says and what happened to it.

### Presentational and container components

The idea, from 2015: split every component in two.

> **Presentational components** are concerned with *how things look*… Have no dependencies on
> the rest of the app, such as Flux actions or stores.
>
> **Container components** are concerned with *how things work*… Provide the data and behavior
> to presentational or other container components.

So `ExpenseList` renders rows and knows nothing; `ExpenseListContainer` fetches, holds state,
and hands data down.

In 2019 the author added a note to the top of that article:

> Update from 2019: I wrote this article a long time ago and my views have since evolved. In
> particular, I don't *suggest* splitting your components like this anymore.

The reason is the previous topic. The split existed to get stateful logic out of a component
that renders — and a custom Hook does that without cutting the component in two. `useExpenses()`
inside `ExpenseList` gives you the separation the container gave you, in one file, with no
prop-forwarding layer in the middle.

> **A word that means two things.** LESSON 5 called `Card` a *container* because it wraps
> `children`. That is unrelated to a *container component* in this pattern. Same word, different
> subject — one is about layout, the other about who owns data.

**What survives** is worth keeping, and it is not a rule about folders or names:

- A component that takes only props and renders them is easier to reuse, easier to read, and
  later easier to test — it has no hidden inputs.
- Fetching, state and side effects are worth *separating from* markup. Where that separation
  now lives is a custom Hook, not a wrapper component.

So the useful question is no longer "is this a container?" but **where does the state live?**

### Answering it properly

React gives an exact procedure, and you already met it at LESSON 29. Now use it deliberately:

> 1. Identify *every* component that renders something based on that state.
> 2. Find their closest common parent component — a component above them all in the hierarchy.
> 3. Decide where the state should live:
>    1. Often, you can put the state directly into their common parent.
>    2. You can also put the state into some component above their common parent.
>    3. If you can't find a component where it makes sense to own the state, create a new
>       component solely for holding the state and add it somewhere in the hierarchy above the
>       common parent component.

Two failure modes sit either side of the right answer.

**Too low** and two components cannot agree — LESSON 25's two counters that would not move
together. You find this one quickly, because the feature does not work.

**Too high** is the one to watch for, because everything works. State that only a menu needs,
parked in `App`, still renders the menu correctly. What it costs is that a state change in `App`
re-renders `App`'s whole subtree by default — every component in the app, to open a menu — and
that every reader of `App` now has to scan a `useState` that has nothing to do with them.

That first cost is LESSON 38's vocabulary, not a new claim: rendering is React calling your
components, and it calls the ones inside them too. You measured it in LESSON 35, where a
component that produced no DOM change was rendered anyway. The example cell counts the
difference in components.

### Not everything is state

Before reaching for `useState`, three earlier lessons rule out cases:

| the value | where it belongs | lesson |
|---|---|---|
| can be computed from state or props | a `const` above the `return` | L29 |
| survives renders but must not cause one | a ref | L50 |
| someone might want to send a link to it | the URL | L61 |

React's own three questions are shorter: does it stay the same over time, is it passed in from a
parent, and can you compute it? Any yes means it is not state.

### Key Notes

- The presentational/container split is history; its author withdrew the recommendation in 2019.
  Custom Hooks replaced it.
- The durable part: separate data-fetching and state from markup — with a Hook, not a wrapper.
- State belongs in the closest common parent of everything that reads it.
- Too low breaks the feature; too high still works, which is why it survives — and re-renders a
  whole subtree to move a menu.

### Example

**Runnable — plain JS.** A component tree, and what "too high" actually costs. Nothing here
imitates React: it counts nodes in a tree, which is exactly what "re-renders its subtree by
default" means in terms of components.

In [ ]:
// L66 — closest common parent, and the cost of going higher

const l66Tree = {
  name: "App",
  children: [
    {
      name: "Header",
      children: [
        { name: "MenuButton", children: [] },
        { name: "MenuList", children: [{ name: "MenuItem", children: [] }] },
      ],
    },
    {
      name: "Directory",
      children: [
        { name: "SearchBox", children: [] },
        {
          name: "Results",
          children: [{ name: "ResultRow", children: [] }, { name: "EmptyState", children: [] }],
        },
      ],
    },
    { name: "Footer", children: [] },
  ],
};

function l66PathTo(node, target, trail = []) {
  const here = [...trail, node.name];
  if (node.name === target) return here;
  for (const child of node.children) {
    const found = l66PathTo(child, target, here);
    if (found) return found;
  }
  return null;
}

function l66ClosestCommonParent(readers) {
  const paths = readers.map((name) => l66PathTo(l66Tree, name));
  let common = [];
  for (let i = 0; i < paths[0].length; i += 1) {
    const segment = paths[0][i];
    if (paths.every((p) => p[i] === segment)) common.push(segment);
    else break;
  }
  return common[common.length - 1];
}

function l66Find(node, name) {
  if (node.name === name) return node;
  for (const child of node.children) {
    const found = l66Find(child, name);
    if (found) return found;
  }
  return null;
}

const l66SubtreeSize = (node) =>
  1 + node.children.reduce((total, child) => total + l66SubtreeSize(child), 0);

// --- a menu flag: only two components read it -------------------------------
const l66MenuReaders = ["MenuButton", "MenuList"];
const l66MenuOwner = l66ClosestCommonParent(l66MenuReaders);

console.log("isMenuOpen is read by:", l66MenuReaders.join(", "));
console.log("  closest common parent:", l66MenuOwner);
console.log("  components re-rendered when it changes:", l66SubtreeSize(l66Find(l66Tree, l66MenuOwner)));
console.log("  if you park it in App instead:", l66SubtreeSize(l66Tree));

// --- a search query: two different branches read it -------------------------
const l66QueryReaders = ["SearchBox", "Results"];
const l66QueryOwner = l66ClosestCommonParent(l66QueryReaders);

console.log("\nquery is read by:", l66QueryReaders.join(", "));
console.log("  closest common parent:", l66QueryOwner);
console.log("  components re-rendered when it changes:", l66SubtreeSize(l66Find(l66Tree, l66QueryOwner)));
console.log("  nothing to lift down — this one genuinely belongs there");

### Exercise

**Runnable — plain JS**, reusing the tree and helpers from the example.

1. The detail panel arrives: add `DetailPanel` under `Directory`, with a child `DetailField`.
   `selectedId` is read by `ResultRow` (to highlight) and by `DetailPanel`. Where does it live,
   and how many components re-render when the selection changes?
2. `MenuItem` now also needs `isMenuOpen` (it closes the menu when clicked). Recompute the
   owner. Did it move?
3. A theme value is read by `MenuList`, `EmptyState` and `Footer`. Compute the owner, then say
   in a comment which of React's three options this is — the common parent, something above it,
   or a new component created solely to hold it — and why Context (LESSON 56) is the usual
   answer for this particular value.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** Four values from an app you have already built. For each, decide
whether it is state, a derived value, a ref, or belongs in the URL — then write
`l66Classify(value)` returning your answer, and print a table.

```
1. the text currently typed in the search box
2. the number of results currently shown
3. the id of the timer used to debounce that search
4. the department the user is filtering by
```

Then, in a comment: number 2 and number 4 are the interesting ones. One is frequently made into
state by mistake, and one is frequently left out of the URL by mistake. Say which is which, and
what goes wrong in each case — for number 4, in terms of what a user can and cannot do with the
browser's Back button and a copied link.

In [ ]:
// Your code here

## LESSON 67 — Designing a component's prop API

A component's props are its **public interface**. Everything else — the state inside it, the
Hooks it calls, the markup it produces — you can change freely. The props are what other files
depend on, and changing them means changing every call site.

That is worth five minutes of thought before you write the component, and this lesson is those
five minutes.

### Name props for what they mean

`<Badge red />` says how it looks. `<Badge tone="danger" />` says what it means. When the design
changes and danger becomes orange, only one of those two needs editing — and only one of them
reads correctly in the call site of a component you wrote a month ago.

### One prop with several values beats several booleans

This is the most common prop-API mistake, and its cost is measurable — the example cell counts
it. Three booleans:

```jsx
<Button isPrimary isSecondary />      // …and what is that supposed to look like?
```

Three booleans have eight combinations, of which most are contradictory. The component then
grows the code to resolve them, and the resolution order becomes an undocumented rule. One
`variant` prop has exactly the number of values you designed, and no impossible state exists to
resolve.

The general principle is React's own advice about state, applied to props: a shape in which two
pieces of information can disagree is a shape that will eventually disagree.

### Prefer `children` to a content prop

If a prop's job is "the stuff inside", it is `children`. `<Card title="Revenue" body="12,400" />`
locks the caller into plain strings; `<Card title="Revenue"><strong>12,400</strong></Card>` does
not. This is LESSON 17 again, and it is the answer to a component sprouting a fourth and fifth
content prop.

### Pass what the component needs

```jsx
<ExpenseRow expense={expense} />              // one prop, coupled to the shape
<ExpenseRow label={expense.title} amount={expense.amount} />   // two props, independent
```

Neither is wrong. The question is whether the component is *about* that object. `ExpenseRow` is
— give it the expense. A generic `<Row>` is not — give it values, and it will still work when
the data comes from somewhere else. What you should avoid is passing a whole object so the
component can pick two fields out of it, because now it only works with that API's shape.

### Spread with restraint

React's wording is exact:

> **Use spread syntax with restraint.** If you're using it in every other component, something
> is wrong. Often, it indicates that you should split your components and pass children as JSX.

`{...props}` is genuinely useful in a thin wrapper around a DOM element — a `Button` that
forwards `onClick`, `type`, `disabled` and `aria-*` to `<button>` without listing them. It is a
warning sign anywhere else, because it makes a component's interface unknowable: you cannot read
the signature and know what it accepts.

### Don't design for the second use before it exists

The last one is a habit rather than a rule. A component used once needs no configuration. The
moment you add `showHeader`, `compact`, `variant` and `align` to a component with one call site,
you have designed an API for callers who may never arrive — and every one of those props is now
a branch to read, and later to keep working.

The rule from LESSON 62 applies here too: **some duplication is fine.** Write the second copy.
When the third arrives, you will know from the three real cases what the shared component
actually needs — which is knowledge you could not have had before.

### Key Notes

- Props are the public interface: cheap to change before, expensive after.
- Name by meaning (`tone="danger"`), not appearance (`red`).
- One `variant` prop beats several booleans — booleans multiply into impossible combinations.
- `children` for content; spread only in a thin wrapper; configure only when a second caller
  actually exists.

### Example

**Runnable — plain JS.** Counting what boolean props cost. The point is not that eight is a big
number — it is which of the eight you would have to write code for.

In [ ]:
// L67 — three booleans versus one variant

const l67Flags = ["isPrimary", "isSecondary", "isDanger"];

function l67Combinations(flags) {
  const rows = [];
  for (let mask = 0; mask < 2 ** flags.length; mask += 1) {
    const combo = {};
    flags.forEach((flag, i) => { combo[flag] = Boolean(mask & (1 << i)); });
    rows.push(combo);
  }
  return rows;
}

const l67All = l67Combinations(l67Flags);
const l67Set = (combo) => Object.keys(combo).filter((k) => combo[k]);

console.log("combinations of", l67Flags.length, "booleans:", l67All.length);

for (const combo of l67All) {
  const on = l67Set(combo);
  const verdict =
    on.length === 0 ? "fine — the default" :
    on.length === 1 ? "fine" :
    "IMPOSSIBLE — which one wins?";
  console.log(` ${(on.join(" + ") || "(none)").padEnd(34)} ${verdict}`);
}

const l67Impossible = l67All.filter((c) => l67Set(c).length > 1).length;
console.log("\nstates the component must somehow resolve:", l67Impossible);

// the same design decision, as one prop
const l67Variants = ["primary", "secondary", "danger"];
console.log("with variant:", l67Variants.length, "values, impossible states:", 0);

// and how it scales
for (const n of [2, 3, 4, 5]) {
  const total = 2 ** n;
  console.log(`  ${n} booleans -> ${total} combinations, ${total - n - 1} impossible`);
}

### Exercise

**Runnable — plain JS.**

A colleague's component is called like this:

```jsx
<UserCard
  user={user}
  showEmail
  showPhone
  hideAvatar
  isCompact
  isLarge
  red
  onClick={…}
/>
```

1. Write `l67Audit(props)` that takes that props object and returns a list of findings, one per
   problem you can detect mechanically: a pair that cannot both be true, a `hideX` next to
   `showY` (mixed polarity), and a prop named for appearance. Run it and print the findings.
2. Redesign the interface. Write it out as a **non-runnable JSX comment** showing the new call
   site, then write `l67Valid(props)` that returns `true` only for combinations your new design
   can express. Show that the impossible combination from part 1 is now unrepresentable.
3. `user={user}` — argue both sides in a comment, in two sentences. When is passing the whole
   object right, and what would make you split it into fields?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**In your project** — and no code cell, because the subject is code you have already written.

Open your Mini-project 3 repository and pick the three components with the most props. For each,
write down:

1. Every prop, and whether its name says *what it means* or *how it looks*.
2. Any two props that could contradict each other, and what your component currently does when
   they do.
3. One prop that exists for a caller that never arrived.

Then change exactly one of the three components, and check every call site still reads clearly.
One component, not three — the point is to see the difference, not to refactor a finished
project.

If you find nothing wrong in any of the three, that is a real result and worth noticing: it
usually means those components were extracted from working code rather than designed up front.

> **Topic 22 complete — LESSON 65, 66 and 67.** You now have the three decisions that turn a
> pile of components into a codebase: where files go, where state lives, and what a component
> accepts. None of them is a React API, and all three are what separates code you can change
> from code you can only add to.
>
> Topic 23 is performance, and it opens with a rule that depends on this topic: **make it
> correct, measure, then optimise.** Almost every "React is slow" problem turns out to be state
> living too high — which you now know how to find.